In [1]:
%load_ext autoreload

In [22]:
%autoreload 2

from JacobianODE.jacobians.jacobian_utils import load_config, initialize_config, make_trajectories, normalize_data, postprocess_data
from JacobianODE.jacobians.run_jacobians import train_jacobians

import numpy as np
import torch


In [14]:
# FILL THIS IN FOR YOUR OWN USE
save_dir = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning"
wandb_entity = "chaotic-consciousness"

# Custom Data Loader Demo

This notebook demonstrates how to use custom data arrays with the JacobianODE framework.

**Data Format**: Custom data must be in the format **Trials x Timepoints x Variables**
- Trials: Number of independent trajectories/trials
- Timepoints: Number of time steps in each trajectory
- Variables: Number of dimensions/variables observed at each time point

In [29]:
# Generate Lorenz data

# load the config
overrides = [
    "data=dysts",
    "data.flow._target_=JacobianODE.dysts_sim.flows.Lorenz",
    "data.postprocessing.obs_noise=0.01",
    f"training.logger.save_dir={save_dir}",
]
cfg = load_config(overrides=overrides)

cfg = initialize_config(cfg)

np.random.seed(cfg.data.flow.random_state)
torch.random.manual_seed(cfg.data.flow.random_state)
eq, sol, dt = make_trajectories(cfg)
custom_data = sol['values']

In [34]:
# Load config with custom dataset loader
cfg = load_config(
    overrides=[
        "data=custom",
        "data.postprocessing.obs_noise=0",
        "data.flow.dim=3",  # Must match n_variables
        "training.lightning.loop_closure_weight=0.001",
        f"training.logger.save_dir={save_dir}",
        f"wandb_entity={wandb_entity}",
    ],
    custom_dataset_loader="JacobianODE.jacobians.dataset_loaders.CustomArrayLoader",
    custom_dataset_loader_kwargs={"data": custom_data, "dt": dt}
)

In [ ]:
values_raw = sol['values']
print(f"Loaded data shape: {values_raw.shape}")  # Should be (Trials, Timepoints, Variables)

# For custom data, we typically don't have alternative noise data
raw_values_noise = None

# ----------------------------------------
# POSTPROCESS DATA
# ----------------------------------------
# Postprocess data (adds noise, filtering, etc. based on config)
values = postprocess_data(cfg, values_raw, dt=dt)
if cfg.data.postprocessing.normalize:
    values, mu, sigma = normalize_data(values)
else:
    mu = 0
    sigma = 1

print(f"Postprocessed data shape: {values.shape}")

At this point, the array `values` is an np.ndarray of shape (Trials, Timepoints, Variables) and is ready to be used for training.


## Next steps

The data is now ready for:
1. Creating dataloaders: `create_dataloaders(cfg, values)`
2. Training: `train_jacobians(cfg)`

In [ ]:
# Example: Continue with full training pipeline
from JacobianODE.jacobians.jacobian_utils import create_dataloaders, setup_wandb, make_model, log_training_info, train_model

# Create dataloaders
train_dataloader, val_dataloader, test_dataloader, trajs = create_dataloaders(cfg, values)

# Set up wandb
name, project, entity = setup_wandb(cfg, trajs, prompt_entity=False)

# Make model
lit_model = make_model(cfg, dt, eq=None, project=project, mu=mu, sigma=sigma, verbose=True)

# Log training info
log_training_info(train_dataloader, trajs, lit_model)

# Train model (uncomment to actually train)
# train_model(cfg, lit_model, train_dataloader, val_dataloader, name, project, entity=entity)
